In [0]:
%sql
USE workspace.colloky;

In [0]:
%sql
CREATE OR REPLACE TABLE colloky_gold_product_lifecycle AS
WITH s AS (
    SELECT * FROM colloky_silver_snapshots WHERE is_complete_run
),
bounds AS (
    SELECT MIN(date_scraped) AS first_run, MAX(date_scraped) AS last_run FROM s
),
first_snap AS (
    SELECT * FROM s
    QUALIFY ROW_NUMBER() OVER (PARTITION BY product_url ORDER BY date_scraped) = 1
),
last_snap AS (
    SELECT * FROM s
    QUALIFY ROW_NUMBER() OVER (PARTITION BY product_url ORDER BY date_scraped DESC) = 1
),
stats AS (
    SELECT product_url,
           COUNT(*)         AS n_snapshots,
           MIN(final_price) AS min_price,
           MAX(final_price) AS max_price
    FROM s
    GROUP BY product_url
)
SELECT
    product_url,
    l.sku,
    l.product_name,
    l.category,
    l.product_type,
    l.gender,
    l.age_segment,
    l.license_character,
    f.date_scraped                    AS first_seen,
    l.date_scraped                    AS last_seen,
    st.n_snapshots,
    f.original_price                  AS first_list_price,
    l.original_price                  AS last_list_price,
    f.final_price                     AS first_price,
    l.final_price                     AS last_price,
    st.min_price,
    st.max_price,
    f.stockout_rate                   AS first_stockout_rate,
    l.stockout_rate                   AS last_stockout_rate,
    l.stockout_rate - f.stockout_rate AS stockout_increase,
    f.date_scraped > b.first_run      AS is_new_arrival,
    l.date_scraped < b.last_run       AS left_catalog
FROM last_snap l
JOIN first_snap f USING (product_url)
JOIN stats st     USING (product_url)
CROSS JOIN bounds b;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE colloky_gold_price_changes AS
WITH s AS (
    SELECT product_url, sku, product_name, category, product_type,
           date_scraped, original_price, final_price, discount
    FROM colloky_silver_snapshots
    WHERE is_complete_run
),
w AS (
    SELECT *,
        LAG(date_scraped)   OVER win AS prev_date,
        LAG(original_price) OVER win AS prev_list_price,
        LAG(final_price)    OVER win AS prev_price,
        LAG(discount)       OVER win AS prev_discount
    FROM s
    WINDOW win AS (PARTITION BY product_url ORDER BY date_scraped)
)
SELECT
    product_url, sku, product_name, category, product_type,
    prev_date        AS changed_from,
    date_scraped     AS changed_at,
    prev_list_price,
    original_price   AS new_list_price,
    prev_price,
    final_price      AS new_price,
    prev_discount,
    discount         AS new_discount,
    CASE
        WHEN original_price > prev_list_price AND final_price >= prev_price THEN 'Inflated reference price'
        WHEN final_price < prev_price  THEN 'Markdown'
        WHEN final_price > prev_price  THEN 'Price increase'
        ELSE 'List price change'
    END AS change_type
FROM w
WHERE prev_date IS NOT NULL
  AND (final_price <> prev_price OR original_price <> prev_list_price);

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE colloky_gold_category_kpis AS
SELECT
    date_scraped,
    snapshot_date,
    category,
    COUNT(DISTINCT product_url)                         AS products,
    ROUND(AVG(discount), 4)                             AS avg_discount,
    ROUND(AVG(CASE WHEN discount > 0 THEN 1 ELSE 0 END), 4) AS pct_on_sale,
    ROUND(AVG(final_price), 0)                          AS avg_price,
    ROUND(AVG(stockout_rate), 4)                        AS avg_stockout_rate
FROM colloky_silver_snapshots
WHERE is_complete_run
GROUP BY ALL;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE colloky_gold_catalog_churn AS
WITH presence AS (
    SELECT DISTINCT date_scraped, product_url
    FROM colloky_silver_snapshots
    WHERE is_complete_run
),
runs AS (
    SELECT date_scraped, LAG(date_scraped) OVER (ORDER BY date_scraped) AS prev_run
    FROM (SELECT DISTINCT date_scraped FROM presence)
),
cur AS (
    SELECT r.date_scraped, p.product_url
    FROM runs r JOIN presence p ON p.date_scraped = r.date_scraped
    WHERE r.prev_run IS NOT NULL
),
prev AS (
    SELECT r.date_scraped, p.product_url
    FROM runs r JOIN presence p ON p.date_scraped = r.prev_run
)
SELECT
    COALESCE(c.date_scraped, p.date_scraped)          AS date_scraped,
    COUNT(CASE WHEN p.product_url IS NULL THEN 1 END) AS products_added,
    COUNT(CASE WHEN c.product_url IS NULL THEN 1 END) AS products_removed
FROM cur c
FULL OUTER JOIN prev p
  ON c.date_scraped = p.date_scraped AND c.product_url = p.product_url
GROUP BY 1
ORDER BY 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT change_type, COUNT(*) AS changes, COUNT(DISTINCT product_url) AS products
FROM colloky_gold_price_changes
GROUP BY change_type
ORDER BY changes DESC;

change_type,changes,products
Price increase,1553,1549
Markdown,1315,1313
Inflated reference price,5,5
List price change,2,2


In [0]:
%sql
SELECT
    changed_at,
    change_type,
    COUNT(*) AS changes,
    ROUND(AVG((new_price - prev_price) / prev_price) * 100, 1) AS avg_price_change_pct
FROM colloky_gold_price_changes
GROUP BY ALL
ORDER BY changed_at, change_type;

changed_at,change_type,changes,avg_price_change_pct
2026-10-06T19:35:00.000Z,Price increase,1,150.0
2026-10-06T20:30:00.000Z,Price increase,1,150.0
2026-10-06T22:27:00.000Z,Price increase,2,150.0
2026-10-06T23:51:00.000Z,Markdown,1,-60.0
2026-10-07T09:24:00.000Z,Markdown,2,-55.0
2026-10-07T09:24:00.000Z,Price increase,1,150.0
2026-10-07T15:06:00.000Z,List price change,2,0.0
2026-10-07T15:06:00.000Z,Markdown,103,-21.6
2026-10-07T15:06:00.000Z,Price increase,72,31.6
2026-10-07T16:01:00.000Z,Inflated reference price,5,10.0


In [0]:
%sql
SELECT
    product_name,
    changed_from,
    changed_at,
    prev_list_price,
    new_list_price,
    prev_price,
    new_price,
    ROUND(prev_discount * 100) AS prev_discount_pct,
    ROUND(new_discount * 100)  AS new_discount_pct
FROM colloky_gold_price_changes
WHERE change_type = 'Inflated reference price'
ORDER BY changed_at;

product_name,changed_from,changed_at,prev_list_price,new_list_price,prev_price,new_price,prev_discount_pct,new_discount_pct
Botín outdoor waterproof niño azul 28-38,2026-10-07T15:06:00.000Z,2026-10-07T16:01:00.000Z,22495,44990,17996,22495,20,50
Botín Ergonómico chelsea niña beige 18-21,2026-10-07T15:06:00.000Z,2026-10-07T16:01:00.000Z,20994,34990,17495,17495,17,50
Bota Ergonómico de agua punta ancha niña celeste 28-33,2026-10-07T15:06:00.000Z,2026-10-07T16:01:00.000Z,9995,19990,7996,9995,20,50
Botín house niña gris 28-38,2026-10-07T15:06:00.000Z,2026-10-07T16:01:00.000Z,19995,39990,19995,19995,0,50
Botín outdoor waterproof niño verde 28-38,2026-10-07T15:06:00.000Z,2026-10-07T16:01:00.000Z,22495,44990,22495,22495,0,50


In [0]:
%sql
WITH s AS (
    SELECT
        product_url,
        date_scraped,
        final_price,
        offers_in_stock,
        LAG(final_price)     OVER w AS prev_price,
        LAG(offers_in_stock) OVER w AS prev_in_stock
    FROM colloky_silver_snapshots
    WHERE is_complete_run
    WINDOW w AS (PARTITION BY product_url ORDER BY date_scraped)
)
SELECT
    CASE WHEN final_price > prev_price THEN 'Price increase' ELSE 'Markdown' END AS change_type,
    CASE
        WHEN offers_in_stock < prev_in_stock THEN 'Fewer sizes in stock'
        WHEN offers_in_stock > prev_in_stock THEN 'More sizes in stock'
        ELSE 'Same sizes in stock'
    END AS stock_movement,
    COUNT(*) AS changes
FROM s
WHERE prev_price IS NOT NULL
  AND final_price <> prev_price
GROUP BY ALL
ORDER BY change_type, changes DESC;

change_type,stock_movement,changes
Markdown,Same sizes in stock,1254
Markdown,More sizes in stock,44
Markdown,Fewer sizes in stock,17
Price increase,Same sizes in stock,1402
Price increase,More sizes in stock,92
Price increase,Fewer sizes in stock,61


In [0]:
%sql
SELECT changed_at, change_type, COUNT(*) AS changes
FROM colloky_gold_price_changes
GROUP BY ALL
ORDER BY changes DESC
LIMIT 10;

changed_at,change_type,changes
2026-10-07T22:07:00.000Z,Price increase,1441
2026-10-07T22:07:00.000Z,Markdown,1196
2026-10-07T15:06:00.000Z,Markdown,103
2026-10-07T15:06:00.000Z,Price increase,72
2026-10-07T23:00:00.000Z,Price increase,20
2026-10-07T16:01:00.000Z,Price increase,6
2026-10-07T16:01:00.000Z,Inflated reference price,5
2026-10-07T17:05:00.000Z,Price increase,5
2026-10-07T23:00:00.000Z,Markdown,5
2026-10-07T19:16:00.000Z,Markdown,3


In [0]:
%sql
WITH c AS (
    SELECT
        product_url,
        changed_at,
        change_type,
        prev_price,
        new_price,
        LEAD(new_price)  OVER (PARTITION BY product_url ORDER BY changed_at) AS next_price,
        LEAD(changed_at) OVER (PARTITION BY product_url ORDER BY changed_at) AS next_change_at
    FROM colloky_gold_price_changes
)
SELECT
    change_type,
    COUNT(*) AS total_changes,
    SUM(CASE WHEN next_price = prev_price THEN 1 ELSE 0 END) AS reverted_later,
    ROUND(AVG(
        CASE WHEN next_price = prev_price
             THEN (unix_timestamp(next_change_at) - unix_timestamp(changed_at)) / 3600
        END
    ), 1) AS avg_hours_until_revert
FROM c
WHERE change_type IN ('Price increase', 'Markdown')
GROUP BY change_type;

change_type,total_changes,reverted_later,avg_hours_until_revert
Price increase,1553,7,4.4
Markdown,1315,47,5.3


In [0]:
%sql
SELECT
    product_name,
    prev_price,
    new_price,
    prev_list_price,
    new_list_price,
    product_url
FROM colloky_gold_price_changes
WHERE changed_at = TIMESTAMP '2026-10-07 22:07:00'
ORDER BY rand()
LIMIT 10;

product_name,prev_price,new_price,prev_list_price,new_list_price,product_url
Calcetín fútbol niño blanco 26/29 - 34/39,4794,3995,7990,7990,https://www.colloky.cl/calcetin-futbol-c74-blanco-unisex-c6peu410i26
Sandalia de Playa Punta Abierta en EVA Azul,5996,7495,14990,14990,https://www.colloky.cl/sandalia-playa-nino-azul-53630449v25
Entero summer vibes filtro uv niño azul 6-9M a 4-5A,17990,8995,17990,17990,https://www.colloky.cl/entero-summer-vibes-filtro-uv-nino-azul-6-9m-a-4-5a
Ballerina lazo niña negra 28-38,17592,10995,21990,21990,https://www.colloky.cl/ballerina-lazo-nina-negra-28-38
Pijama algodón Entero Blanco Estampado Niño,7196,8995,17990,17990,https://www.colloky.cl/pijama-tematico-dormir-nino-blanco-pjje0578v25
Zapatilla urbana Dubbo unisex blanco 28-38,17994,14995,29990,29990,https://www.colloky.cl/zapatilla-urbana-unisex-blanco-28-38
Calcetin antideslizante Marvel Spiderman niño gris 23-39,6293,4495,8990,8990,https://www.colloky.cl/calcetin-antideslizante-marvel-spiderman-nino-gris-3-4a-a-9-12a
Set body rib Disney Mickey Mouse niño denim 0/3M a 18/24M,19990,9995,19990,19990,https://www.colloky.cl/set-body-rib-disney-mickey-mouse-nino-denim-0-3m-a-18-24m
Polera de algodón Corta con hombro caído,4396,5495,10990,10990,https://www.colloky.cl/polera-every-day-nina-beige-p1je8675v25
Sandalia estampada niña multicolor 21-27,10392,6495,12990,12990,https://www.colloky.cl/sandalia-estampada-nina-multicolor-21-27


In [0]:
%sql
SELECT
    date_scraped,
    ROUND(discount * 100) AS discount_pct,
    COUNT(*)              AS products
FROM colloky_silver_snapshots
WHERE date_scraped >= TIMESTAMP '2026-10-07 19:00:00'
GROUP BY ALL
ORDER BY date_scraped, products DESC;

date_scraped,discount_pct,products
2026-10-07T19:16:00.000Z,60,1416
2026-10-07T19:16:00.000Z,50,669
2026-10-07T19:16:00.000Z,40,387
2026-10-07T19:16:00.000Z,20,343
2026-10-07T19:16:00.000Z,30,286
2026-10-07T19:16:00.000Z,0,183
2026-10-07T19:16:00.000Z,70,11
2026-10-07T19:16:00.000Z,71,8
2026-10-07T19:16:00.000Z,62,7
2026-10-07T19:16:00.000Z,57,7


In [0]:
%sql
SELECT
    date_scraped,
    COUNT(*) AS products,
    ROUND(AVG(CASE WHEN ROUND(discount * 100) = 50 THEN 1 ELSE 0 END) * 100, 1) AS pct_at_50_off,
    ROUND(AVG(CASE WHEN ROUND(discount * 100) = 60 THEN 1 ELSE 0 END) * 100, 1) AS pct_at_60_off,
    ROUND(AVG(CASE WHEN discount = 0 THEN 1 ELSE 0 END) * 100, 1)                AS pct_full_price,
    ROUND(AVG(discount) * 100, 1)                                                 AS avg_discount_pct
FROM colloky_silver_snapshots
WHERE date_scraped >= TIMESTAMP '2026-10-07 15:00:00'
GROUP BY date_scraped
ORDER BY date_scraped;

date_scraped,products,pct_at_50_off,pct_at_60_off,pct_full_price,avg_discount_pct
2026-10-07T15:06:00.000Z,3257,20.2,41.8,5.6,45.5
2026-10-07T16:01:00.000Z,3266,20.3,41.9,5.7,45.5
2026-10-07T17:05:00.000Z,3314,20.1,42.4,5.6,45.7
2026-10-07T18:01:00.000Z,3312,20.1,42.5,5.5,45.7
2026-10-07T19:16:00.000Z,3328,20.1,42.5,5.5,45.7
2026-10-07T20:14:00.000Z,3316,20.2,42.3,5.5,45.7
2026-10-07T21:01:00.000Z,3313,20.2,42.3,5.5,45.7
2026-10-07T22:07:00.000Z,3322,99.6,0.0,0.1,50.0
2026-10-07T23:00:00.000Z,3337,99.3,0.0,0.4,49.8


In [0]:
%sql
CREATE OR REPLACE TABLE colloky_gold_discount_mix_by_run AS
SELECT
    date_scraped,
    COUNT(*) AS products,
    ROUND(AVG(CASE WHEN ROUND(discount * 100) = 50 THEN 1 ELSE 0 END) * 100, 1) AS pct_at_50_off,
    ROUND(AVG(CASE WHEN ROUND(discount * 100) = 60 THEN 1 ELSE 0 END) * 100, 1) AS pct_at_60_off,
    ROUND(AVG(CASE WHEN discount = 0 THEN 1 ELSE 0 END) * 100, 1)                AS pct_full_price,
    ROUND(AVG(discount) * 100, 1)                                                 AS avg_discount_pct
FROM colloky_silver_snapshots
WHERE is_complete_run
GROUP BY date_scraped;

num_affected_rows,num_inserted_rows
